# F — Ngô: PlantVillage → Corn-Leaf-Diseases (CLD), giao thức MSUN

Đặt cạnh **Wu et al. 2023 (MSUN)**, Plant Phenomics 5:0038, **Table 7** (PVD → CLD). Đây là bài **duy nhất** tìm được báo cáo
UDA trên CLD; bài review Hu et al. 2025 (arXiv 2506.05972) chỉ chép lại Table 7.

| | MSUN (PVD → CLD) | Notebook này |
|---|---|---|
| Target | CLD, "4 188 ảnh, 4 lớp", lấy từ Kaggle (bài không ghi tên tập) | Kaggle `smaranjitghose/corn-or-maize-leaf-disease-dataset`: gray leaf spot 574 / blight 1146 / healthy 1162 / rust 1306 = 4 188 |
| Source | PV ngô 411 / 788 / 930 / 954 | **cân bằng** `MAX_PER_CLASS` = 300 / lớp (như A/B/D), qua FBR |
| Đánh giá | transductive | transductive, **thêm** accuracy trên phần target không trùng PV |
| Backbone | ResNet-50 | ResNet-50 |

### Cảnh báo: CLD chứa ảnh PlantVillage

Số ảnh của tập Kaggle khớp với PlantVillage: healthy 1 162 bằng đúng PV `Corn_(maize)___healthy`; blight, rust và gray leaf spot
lớn hơn PV một ít. Nhiều khả năng tập được ghép từ PlantVillage và một phần ảnh khác, tức target của MSUN phần lớn là **ảnh lab**.
Điều này giải thích vì sao MSUN đạt 96.78% và lớp rust đạt 100% ở mọi phương pháp. §3.3 kiểm tra bằng dHash, ghi danh sách
ảnh trùng, và notebook báo **hai** con số: toàn bộ target (so được với MSUN) và **field-only** (bỏ ảnh trùng PV, con số trung thực).
Chưa chạy kiểm tra thì chưa nên coi F là benchmark lab → field.

### Chạy

1. Đặt `KAGGLE_API_TOKEN` trong Colab Secrets (giống notebook C).
2. §2: kiểm tra `SAVE_DIR`, `BACKBONE`. Chạy §1 → §3.3, đọc kỹ số ảnh trùng PV ở §3.3.
3. §7 kiểm tra ghép cặp RNG, §8 huấn luyện, §9 so sánh với Table 7.

## 1 — Setup & Reproducibility

In [ ]:
!pip install segment-anything -q        # SAM cho FBR

import os, random, copy, json, hashlib, warnings, csv, math, collections
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as tv_models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


def set_seed(seed: int = 42):
    # Seed moi nguon ngau nhien (Python, NumPy, PyTorch CPU/GPU) + cuDNN tat dinh.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Generator rieng cho shuffle cua DataLoader. set_seed() KHONG dung toi g,
# nen vong lap huan luyen re-seed g moi epoch (xem train_arm).
g = torch.Generator()
g.manual_seed(42)
print('Setup complete.')

## 2 — Drive, đường dẫn & cấu hình

`SAVE_DIR` dùng chung với notebook A–D. Mọi thứ riêng của F nằm trong `TASK_DIR`. Lớp của CLD được nhận theo tên thư mục
(`CLD_CLASS_KEYWORDS`), nên không phụ thuộc cấu trúc file zip của Kaggle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── SAM cho FBR: 'vit_b' | 'vit_h'. Quyet dinh cache FBR va thu muc ket qua / checkpoint ──
SAM_MODEL_TYPE = 'vit_h'

# Dung chung thu muc goc voi A-D -> PlantVillage da clone san o day.
SAVE_DIR = '/content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/MUSN Apple'

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')   # so khop khong phan biet hoa thuong

TASK        = 'corn_cld'
# Thu tu lop = thu tu cot cua MSUN Table 7
CLASS_NAMES = ['gray_spot', 'blight', 'healthy', 'rust']
NUM_CLASSES = len(CLASS_NAMES)
PVD_FOLDERS = {'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 0, 'Corn_(maize)___Northern_Leaf_Blight': 1,
               'Corn_(maize)___healthy': 2, 'Corn_(maize)___Common_rust_': 3}

# ── CLD (Kaggle) ─────────────────────────────────────────────────────────────
CLD_KAGGLE_SLUG = 'smaranjitghose/corn-or-maize-leaf-disease-dataset'
# Tu khoa trong TEN thu muc -> lop (Blight / Common_Rust / Gray_Leaf_Spot / Healthy).
CLD_CLASS_KEYWORDS = {
    'gray_spot': ('gray', 'grey', 'cercospora'),
    'blight':    ('blight',),
    'healthy':   ('healthy',),
    'rust':      ('rust',),
}

# So anh theo lop cua tap Kaggle (tong 4188 = so MSUN ghi trong bai).
# MSUN Table 3 ghi 1604 / 370 / 1162 / 1411 (tong 4547): ba so chep nham tu bang tao (Table 2) -> khong dung.
REF_TARGET_COUNTS = {'gray_spot': 574, 'blight': 1146, 'healthy': 1162, 'rust': 1306}
# MSUN Table 7: accuracy (%) theo lop, thu tu CLASS_NAMES, va cot "Average". ResNet-50, transductive.
REF_TABLE_NAME = 'MSUN TABLE 7 (PVD -> CLD)'
REF_RESULTS = {
    'Baseline w/o DA [MSUN]': ([61.49, 86.75, 79.05, 99.18], 75.96),
    'DAN [MSUN]':             ([68.29, 91.50, 81.33, 100.0], 87.89),
    'DANN [MSUN]':            ([76.66, 91.73, 82.64, 100.0], 89.47),
    'D-Coral [MSUN]':         ([79.44, 91.88, 82.64, 100.0], 89.90),
    'DAAN [MSUN]':            ([78.57, 92.27, 84.29, 100.0], 90.35),
    'DSAN [MSUN]':            ([81.71, 91.58, 83.42, 100.0], 90.69),
    'MRAN [MSUN]':            ([79.22, 92.05, 83.61, 100.0], 91.06),
    'MSUN [MSUN]':            ([80.09, 93.81, 84.77, 100.0], 96.78),
}

# ── Giao thuc ────────────────────────────────────────────────────────────────
PROTOCOL = 'transductive'    # MSUN: target = danh gia = toan bo CLD
BACKBONE = 'resnet50'        # MSUN dung ResNet-50

# ── Kiem tra anh trung PlantVillage (§3.3) ───────────────────────────────────
DHASH_SIZE      = 8          # dHash 8x8 = 64 bit
DUP_HAMMING_MAX = 4          # <= 4 / 64 bit khac nhau -> coi la ban sao
# True: khong lay anh PV co ban sao trong target vao source (khong ro nhan, nhung lech giao thuc MSUN).
# Neu ca lop healthy cua CLD la anh PV thi True lam lop do het anh source -> §3.3 se dung lai.
EXCLUDE_TARGET_DUPS_FROM_SOURCE = False

# ── Du lieu ──────────────────────────────────────────────────────────────────
DATA_SEED     = 42      # co dinh: lay mau source, split, FBR cache
MAX_PER_CLASS = 300     # source can bang nhu A/B/D
SRC_VAL_RATIO = 0.25    # PV train/val 75/25 (nhu A/B/D)
BATCH_SIZE    = 32      # nhu D

# ── Huan luyen ───────────────────────────────────────────────────────────────
SEEDS      = [42]       # nen chay 3-5 seed
MAX_EPOCHS = 300
MIN_SEL    = 250        # chi chon model tu epoch nay (nhu A/B/D)
LR, WD     = 1e-3, 0.01
LOG_EVERY  = 10         # cung la chu ky luu checkpoint

# ── Duong dan ────────────────────────────────────────────────────────────────
PLANT_VILLAGE_SAVE_PATH = os.path.join(SAVE_DIR, 'PlantVillage')
PLANT_VILLAGE_DIR       = os.path.join(PLANT_VILLAGE_SAVE_PATH, 'raw', 'color')

TASK_DIR = os.path.join(SAVE_DIR, 'cld_corn')
CLD_ROOT = os.path.join(TASK_DIR, 'cld_raw')

_RUN_TAG      = f'{SAM_MODEL_TYPE}_{BACKBONE}_{PROTOCOL}'
# Pool nen va source phu thuoc cong tac loai anh trung -> cache rieng.
FBR_CACHE_DIR = os.path.join(TASK_DIR, f'fbr_{SAM_MODEL_TYPE}_d{DATA_SEED}_n{MAX_PER_CLASS}'
                             + ('_srcdedup' if EXCLUDE_TARGET_DUPS_FROM_SOURCE else ''))
CKPT_DIR      = os.path.join(TASK_DIR, f'checkpoints_{_RUN_TAG}')
RESULTS_DIR   = os.path.join(TASK_DIR, f'results_{_RUN_TAG}')
for _d in (SAVE_DIR, TASK_DIR, CLD_ROOT, FBR_CACHE_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)

print(f'TASK={TASK}  classes={CLASS_NAMES}  protocol={PROTOCOL}  backbone={BACKBONE}')
print(f'TASK_DIR = {TASK_DIR}')

## 3 — Tải dữ liệu

PlantVillage clone từ GitHub như D. CLD tải bằng Kaggle CLI; token **không** ghi trong notebook mà đọc từ Colab Secrets
(`KAGGLE_API_TOKEN`), giống notebook C. Đã có ảnh trong `CLD_ROOT` thì bỏ qua bước tải.

In [ ]:
!pip install kaggle -q


def _clone_if_missing(url, dest):
    if os.path.isdir(dest) and os.listdir(dest):
        print(f'{os.path.basename(dest)} da co, bo qua.')
        return
    os.makedirs(dest, exist_ok=True)
    print(f'Cloning {url} ...')
    !git clone "{url}" "{dest}" --depth=1 -q
    print('Xong.')


_clone_if_missing('https://github.com/spMohanty/PlantVillage-Dataset.git', PLANT_VILLAGE_SAVE_PATH)

_n_img = sum(f.lower().endswith(IMAGE_EXTS) for _, _, fs in os.walk(CLD_ROOT) for f in fs)
if _n_img:
    print(f'CLD da co ({_n_img} anh), bo qua.')
else:
    # Token KHONG ghi trong notebook: dat KAGGLE_API_TOKEN trong Colab Secrets (hoac bien moi truong).
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        KAGGLE_API_TOKEN = os.environ.get('KAGGLE_API_TOKEN', '')
    assert KAGGLE_API_TOKEN, 'Thieu KAGGLE_API_TOKEN (Colab Secrets hoac bien moi truong)'
    _kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(_kaggle_dir, exist_ok=True)
    with open(os.path.join(_kaggle_dir, 'access_token'), 'w') as _fh:
        _fh.write(KAGGLE_API_TOKEN)
    !chmod 600 {_kaggle_dir}/access_token
    print(f'Downloading {CLD_KAGGLE_SLUG} ...')
    !kaggle datasets download -d {CLD_KAGGLE_SLUG} -p "{CLD_ROOT}" --unzip -q
    _n_img = sum(f.lower().endswith(IMAGE_EXTS) for _, _, fs in os.walk(CLD_ROOT) for f in fs)
    assert _n_img, f'Tai CLD that bai: khong co anh nao trong {CLD_ROOT}'
    print(f'Xong: {_n_img} anh.')

### 3.1 — Transforms & Dataset class (giống hệt notebook D)

In [ ]:
_NORM = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TRAIN_TRANSFORM = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    _NORM,
])

VAL_TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    _NORM,
])


class PlantDiseaseDataset(Dataset):
    # samples: list (path, label). labeled=False -> tra ve -1 thay cho nhan,
    # chan ro ri nhan target vao huan luyen.
    def __init__(self, samples, classes, transform=None, labeled=True):
        self.samples      = samples
        self.classes      = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform    = transform
        self.labeled      = labeled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return (image, label) if self.labeled else (image, -1)

    def get_labels(self):
        return [s[1] for s in self.samples]

print('Transforms and dataset class defined.')

### 3.2 — Nạp CLD theo tên thư mục lớp

Bảng in ra liệt kê mọi thư mục có ảnh, kèm lớp được gán, rồi so số ảnh từng lớp với tập Kaggle (574 / 1146 / 1162 / 1306).
Ảnh trùng MD5 chỉ được giữ một bản.

In [ ]:
# === CELL-TAG: data_funcs ===
VALID_EXTS = IMAGE_EXTS                            # §2
_RESAMPLE  = getattr(Image, 'Resampling', Image)   # Pillow >= 9.1 dat hang so trong Image.Resampling


def _find_folder(root, name):
    # Ten thu muc lech nhau giua cac ban phat hanh -> fallback khong phan biet hoa thuong.
    exact = os.path.join(root, name)
    if os.path.isdir(exact):
        return exact
    for f in os.listdir(root):
        if f.lower() == name.lower() or name.lower() in f.lower():
            return os.path.join(root, f)
    return None


def _list_images(folder):
    return sorted(os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(VALID_EXTS))


def pvd_class_images(data_dir, class_map):
    # Moi anh PV cua cac lop dung trong bai (chua lay mau): {label: [path, ...]}.
    out = {}
    for folder_name, label in class_map.items():
        folder = _find_folder(data_dir, folder_name)
        if folder is None:
            raise FileNotFoundError(f'Khong thay thu muc PV: {folder_name} trong {data_dir}')
        out[label] = _list_images(folder)
    return out


def load_pvd_samples(pv_images, max_per_class, seed, exclude=frozenset()):
    # Lay toi da max_per_class anh moi lop, xao tron bang RNG rieng (khong dung RNG toan cuc).
    # exclude: anh PV bi loai truoc khi lay mau (vd anh trung voi target, xem §3.3).
    rng = random.Random(seed)
    samples = []
    print('[PVD] Source (lab):')
    for label, imgs in pv_images.items():
        pool = [p for p in imgs if p not in exclude]
        rng.shuffle(pool)
        selected = pool[:max_per_class]
        samples += [(p, label) for p in selected]
        print(f'  {CLASS_NAMES[label]:<16}: {len(imgs):>5} co, {len(imgs) - len(pool):>5} bi loai '
              f'-> {len(selected):>4} dung')
    print(f'  Tong PVD: {len(samples)}')
    return samples


def _md5(path):
    with open(path, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()


def discover_target(root, class_keywords, path_must_contain=(), skip_dir_keywords=()):
    """Tim anh target theo ten thu muc lop, khong phu thuoc cau truc giai nen.

    Moi thu muc chua anh duoc gan cho lop co tu khoa xuat hien trong TEN thu muc (khong phan biet hoa
    thuong). Thu muc khong khop lop nao (vd powdery mildew cua TDTL) bi bo qua va duoc in ra.

    Args:
        root: thu muc goc cua tap target.
        class_keywords: {ten lop: (tu khoa, ...)}, thu tu lop = CLASS_NAMES.
        path_must_contain: chi xet thu muc co DU moi chuoi nay trong duong dan tuong doi voi root
            (vd ('taiwan', 'data augmentation', 'train')).
        skip_dir_keywords: bo thu muc co mot trong cac chuoi nay trong duong dan tuong doi.

    Returns:
        (samples, report): samples = [(path, label)], anh trung MD5 chi giu ban dau tien;
        report = {'dirs': [(rel_dir, lop | None, so anh)], 'md5_duplicates': so anh trung bi bo}.
    """
    samples, dirs, seen, n_dup = [], [], set(), 0
    for dirpath, dirnames, files in os.walk(root):
        dirnames.sort()
        rel = os.path.relpath(dirpath, root)
        if not all(k.lower() in rel.lower() for k in path_must_contain):
            continue
        if any(k.lower() in rel.lower() for k in skip_dir_keywords):
            continue
        imgs = sorted(f for f in files if f.lower().endswith(VALID_EXTS))
        if not imgs:
            continue
        name = os.path.basename(dirpath).lower()
        hits = [c for c, kws in class_keywords.items() if any(k in name for k in kws)]
        if len(hits) > 1:
            raise ValueError(f'Thu muc {rel} khop nhieu lop {hits} -> sua *_CLASS_KEYWORDS o §2')
        dirs.append((rel, hits[0] if hits else None, len(imgs)))
        if not hits:
            continue
        label = CLASS_NAMES.index(hits[0])
        for f in imgs:
            p = os.path.join(dirpath, f)
            h = _md5(p)
            if h in seen:
                n_dup += 1
                continue
            seen.add(h)
            samples.append((p, label))
    return samples, {'dirs': dirs, 'md5_duplicates': n_dup}


def print_target_report(title, samples, report, ref_counts, ref_name):
    print(title)
    for rel, cname, n in report['dirs']:
        print(f'  {"(bo qua)" if cname is None else cname:<16}{n:>6}  {rel}')
    if report['md5_duplicates']:
        print(f'  anh trung MD5 bi bo: {report["md5_duplicates"]}')
    c = np.bincount([l for _, l in samples], minlength=NUM_CLASSES)
    print(f'\n  {"lop":<16}{"so anh":>8}{ref_name:>14}{"chenh":>8}')
    for i, cname in enumerate(CLASS_NAMES):
        print(f'  {cname:<16}{c[i]:>8}{ref_counts[cname]:>14}{c[i] - ref_counts[cname]:>+8}')
    print(f'  {"tong":<16}{len(samples):>8}{sum(ref_counts.values()):>14}'
          f'{len(samples) - sum(ref_counts.values()):>+8}')


# ── Phat hien anh target trung voi PlantVillage (dHash) ───────────────────────────────────────
def dhash(path, size=DHASH_SIZE):
    # Difference hash 64 bit: ben voi resize / nen lai JPEG, KHONG ben voi xoay / lat.
    img = Image.open(path).convert('L').resize((size + 1, size), _RESAMPLE.BILINEAR)
    a = np.asarray(img, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int(np.packbits(bits).view('>u8')[0])


def hash_images(paths, cache_path):
    # dHash cho moi anh, cache JSON tren Drive (doc anh tu Drive cham).
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path) as fh:
            cache = json.load(fh)
    todo = [p for p in paths if p not in cache]
    for p in tqdm(todo, desc=os.path.basename(cache_path), disable=not todo):
        cache[p] = dhash(p)
    if todo:
        with open(cache_path, 'w') as fh:
            json.dump(cache, fh)
    return np.array([cache[p] for p in paths], dtype=np.uint64)


def _popcount64(x):
    return np.unpackbits(x.view(np.uint8)).reshape(-1, 64).sum(1)


def find_pv_duplicates(target_paths, target_hashes, pv_paths, pv_hashes, max_dist):
    """Ghep moi anh target voi anh PV gan nhat theo khoang cach Hamming cua dHash.

    Returns:
        dict {target_path: (pv_path, khoang cach)} cho cac anh target co khoang cach <= max_dist,
        tuc la ban sao (co the da resize / nen lai) cua mot anh PlantVillage.
    """
    dups = {}
    for p, h in zip(target_paths, target_hashes):
        d = _popcount64(pv_hashes ^ h)
        j = int(d.argmin())
        if d[j] <= max_dist:
            dups[p] = (pv_paths[j], int(d[j]))
    return dups


def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                      worker_init_fn=seed_worker, generator=g)


def _count(samples):
    c = np.bincount([s[1] for s in samples], minlength=NUM_CLASSES)
    return ', '.join(f'{n}={k}' for n, k in zip(CLASS_NAMES, c))

print('Data / dedup functions defined.')

In [ ]:
# === CELL-TAG: load_target ===
target_all, _rep = discover_target(CLD_ROOT, CLD_CLASS_KEYWORDS)
print_target_report('CLD - thu muc -> lop:', target_all, _rep, REF_TARGET_COUNTS, 'Kaggle')
assert target_all, 'Khong tim thay anh target nao: kiem tra CLD_CLASS_KEYWORDS'
assert min(np.bincount([l for _, l in target_all], minlength=NUM_CLASSES)) > 0, 'Co lop target khong co anh nao'

### 3.3 — Kiểm tra ảnh trùng PlantVillage & chia dữ liệu

| Tập | Nguồn | Dùng nhãn? | Vai trò |
|---|---|---|---|
| source train / val | PV, tối đa `MAX_PER_CLASS` ảnh/lớp (cân bằng như A/B/D), chia 75/25 | có | `L_cls` / chọn model (val loss) |
| target | **toàn bộ** tập target | **không** | `L_cdan`, `L_cons` |
| test | **chính target** (transductive, như MSUN và CBPUDA) | chỉ để đánh giá | accuracy báo cáo |
| test field-only | test bỏ các ảnh là bản sao của ảnh PV | chỉ để đánh giá | số phụ, không có trong bài báo |
| pool ảnh nền FBR | ảnh target field-only | không | nền thật |

**Vì sao phải kiểm tra ảnh trùng:** một số tập trên Kaggle được ghép lại từ PlantVillage. Một ảnh target là bản sao của ảnh PV thì
(1) là ảnh lab, không phải ảnh field; (2) nếu đúng ảnh đó được lấy vào source thì nhãn của nó rò sang tập đánh giá. Cell dưới tính
dHash (64 bit) của mọi ảnh PV thuộc các lớp đang dùng và mọi ảnh target, coi là trùng khi khoảng cách Hamming ≤ `DUP_HAMMING_MAX`,
và ghi danh sách vào `pv_duplicates.csv`. dHash không bắt được bản đã xoay / lật, nên số trùng in ra là **cận dưới**.

`EXCLUDE_TARGET_DUPS_FROM_SOURCE` (§2): `False` giữ source như MSUN; `True` lấy mẫu source chỉ từ các ảnh PV không có bản sao trong
target (không rò nhãn, nhưng lệch giao thức MSUN).

In [ ]:
# === CELL-TAG: split ===
pv_images = pvd_class_images(PLANT_VILLAGE_DIR, PVD_FOLDERS)
_pv_paths = [p for imgs in pv_images.values() for p in imgs]
_tg_paths = [p for p, _ in target_all]
_pv_hash  = hash_images(_pv_paths, os.path.join(TASK_DIR, 'dhash_pv.json'))
_tg_hash  = hash_images(_tg_paths, os.path.join(TASK_DIR, 'dhash_target.json'))
pv_dups   = find_pv_duplicates(_tg_paths, _tg_hash, _pv_paths, _pv_hash, DUP_HAMMING_MAX)

_dup_c = np.bincount([l for p, l in target_all if p in pv_dups], minlength=NUM_CLASSES)
print(f'Anh target trung voi PV (dHash, Hamming <= {DUP_HAMMING_MAX}): {len(pv_dups)} / {len(target_all)}')
print('  theo lop: ' + ', '.join(f'{c}={n}' for c, n in zip(CLASS_NAMES, _dup_c)))
if pv_dups:
    print('  CANH BAO: cac anh nay la anh LAB nam trong target. Accuracy tren toan bo target (giao thuc MSUN)')
    print('  bi thoi phong; xem them accuracy tren phan "field-only" (§8, §9).')
    with open(os.path.join(TASK_DIR, 'pv_duplicates.csv'), 'w', newline='') as _fh:
        _w = csv.writer(_fh)
        _w.writerow(['target_path', 'pv_path', 'hamming'])
        for _p, (_q, _d) in sorted(pv_dups.items()):
            _w.writerow([_p, _q, _d])

# Source: anh PV trung voi target co the bi loai truoc khi lay mau (cong tac o §2).
_pv_excl = {q for q, _ in pv_dups.values()} if EXCLUDE_TARGET_DUPS_FROM_SOURCE else frozenset()
pvd_samples = load_pvd_samples(pv_images, MAX_PER_CLASS, DATA_SEED, exclude=_pv_excl)
assert min(np.bincount([l for _, l in pvd_samples], minlength=NUM_CLASSES)) > 0, \
    'Mot lop source khong con anh nao sau khi loai anh trung -> dat EXCLUDE_TARGET_DUPS_FROM_SOURCE = False'

# Transductive (MSUN, CBPUDA): target khong nhan = tap danh gia = toan bo anh target.
target_samples = test_samples = target_all
field_test_samples = [s for s in test_samples if s[0] not in pv_dups]

# Pool anh nen FBR: anh target KHONG phai ban sao PV (anh lab co nen phang, khong phai nen field).
FBR_BG_PATHS = sorted(p for p, _ in field_test_samples)
assert FBR_BG_PATHS, 'pool anh nen rong: moi anh target deu trung PV'

# Source train/val: cung chi so cho ban PV goc va ban FBR (FBR cache 1-1 voi pvd_samples).
src_tr_idx, src_vl_idx = train_test_split(
    list(range(len(pvd_samples))), test_size=SRC_VAL_RATIO,
    stratify=[s[1] for s in pvd_samples], random_state=DATA_SEED,
)

print(f'\n{"=" * 64}\nDATA SUMMARY ({TASK}, {PROTOCOL})\n{"=" * 64}')
print(f'  PV  train : {len(src_tr_idx):>5}  ({_count([pvd_samples[i] for i in src_tr_idx])})')
print(f'  PV  val   : {len(src_vl_idx):>5}  ({_count([pvd_samples[i] for i in src_vl_idx])})')
print(f'  target    : {len(target_samples):>5}  ({_count(target_samples)})  KHONG dung nhan')
print(f'  test      : {len(test_samples):>5}  = target (transductive)')
print(f'  field-only: {len(field_test_samples):>5}  ({_count(field_test_samples)})  test bo anh trung PV')
print(f'  Pool nen FBR: {len(FBR_BG_PATHS)} anh target (field-only)')
print('=' * 64)

## 4 — FBR: Field-adaptive Background Recomposition

Giống notebook D: SAM (prompt điểm giữa) → `refine_mask` → ghép lá lên một mảnh ảnh nền field. **Pool ảnh nền = ảnh target
field-only** (§3.3): ảnh target là bản sao ảnh PV có nền phẳng của phòng lab nên bị loại khỏi pool. Vì giao thức là transductive,
pool là ảnh của chính tập đánh giá (cùng lưu ý như B và D). Ảnh nền đọc bằng PIL.

In [ ]:
import cv2

# ── Download and load SAM (SAM_MODEL_TYPE dat o muc 2) ────────────────────────
SAM_FILES = {
    'vit_b': ('sam_vit_b_01ec64.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'),
    'vit_h': ('sam_vit_h_4b8939.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'),
}
_sam_file, _sam_url = SAM_FILES[SAM_MODEL_TYPE]
SAM_CHECKPOINT = f'/content/{_sam_file}'

if not os.path.exists(SAM_CHECKPOINT):
    print(f'Downloading SAM checkpoint ({SAM_MODEL_TYPE})...')
    !wget -q {_sam_url} -O {SAM_CHECKPOINT}.part && mv {SAM_CHECKPOINT}.part {SAM_CHECKPOINT}
    print('SAM checkpoint downloaded.')

from segment_anything import sam_model_registry, SamPredictor
sam      = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT).to(DEVICE)
sam_pred = SamPredictor(sam)
print(f'SAM {SAM_MODEL_TYPE} loaded on {DEVICE}.')

In [ ]:
def _read_rgb(path):
    return np.array(Image.open(path).convert('RGB'))


def segment_leaf(predictor, image_rgb):
    # SAM voi 1 diem prompt o tam anh (la PV luon nam giua).
    h, w = image_rgb.shape[:2]
    predictor.set_image(image_rgb)
    masks, scores, _ = predictor.predict(
        point_coords=np.array([[w // 2, h // 2]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    return masks[np.argmax(scores)]


def refine_mask(mask, k_d=15, k_e=10, k_g=5, iterations=2):
    # Dilate -> erode (x2) roi Gaussian blur: lap lo, lam muot bien, alpha mem.
    m  = mask.astype(np.uint8) * 255
    kd = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_d, k_d))
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_e, k_e))
    for _ in range(iterations):
        m = cv2.dilate(m, kd)
        m = cv2.erode(m, ke)
    m = cv2.GaussianBlur(m, (k_g, k_g), 0)
    return m.astype(np.float32) / 255.0


def composite(lab_img, mask, bg_img):
    h, w  = lab_img.shape[:2]
    bg    = cv2.resize(bg_img, (w, h))
    alpha = mask[:, :, np.newaxis]
    out   = alpha * lab_img.astype(np.float32) + (1 - alpha) * bg.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)


def random_field_background(bg_path, out_w, out_h):
    # Crop ngau nhien 30-70% canh ngan -> xoay ngau nhien -> center-crop 1/sqrt(2) (bo goc den).
    img  = _read_rgb(bg_path)
    H, W = img.shape[:2]
    s = int(random.uniform(0.30, 0.70) * min(H, W))
    x = random.randint(0, W - s)
    y = random.randint(0, H - s)
    patch = img[y:y + s, x:x + s]
    M = cv2.getRotationMatrix2D((s / 2, s / 2), random.uniform(0, 360), 1.0)
    patch = cv2.warpAffine(patch, M, (s, s))
    c = int(s / np.sqrt(2))
    o = (s - c) // 2
    patch = patch[o:o + c, o:o + c]
    return cv2.resize(patch, (out_w, out_h))


def fbr_single(lab_path, bg_paths, predictor):
    lab_rgb = _read_rgb(lab_path)
    h, w    = lab_rgb.shape[:2]
    mask    = refine_mask(segment_leaf(predictor, lab_rgb))
    bg_rgb  = random_field_background(random.choice(bg_paths), w, h)
    return composite(lab_rgb, mask, bg_rgb)

print('FBR functions defined.')

In [ ]:
# ── Pre-compute FBR cache (idempotent: chay lai chi tinh anh con thieu) ─────
fbr_bg_paths = FBR_BG_PATHS                            # anh target field-only (§3.3)
print(f'Pool anh nen (target): {len(fbr_bg_paths)} anh')
print(f'FBR cho {len(pvd_samples)} anh source -> {FBR_CACHE_DIR}')

set_seed(DATA_SEED)   # chon nen + goc xoay tai lap duoc
fbr_samples = []
n_new = n_skip = n_fail = 0
for idx, (orig_path, label) in enumerate(tqdm(pvd_samples)):
    cache_path = os.path.join(FBR_CACHE_DIR, f'{idx:05d}_c{label}.jpg')
    if os.path.exists(cache_path):
        n_skip += 1
    else:
        try:
            Image.fromarray(fbr_single(orig_path, fbr_bg_paths, sam_pred)).save(cache_path, quality=92)
        except Exception as e:
            # Khong am tham: dem lai de biet bao nhieu anh roi ve ban goc.
            print(f'  [fallback] {os.path.basename(orig_path)}: {e}')
            Image.open(orig_path).convert('RGB').save(cache_path, quality=92)
            n_fail += 1
        n_new += 1
    fbr_samples.append((cache_path, label))

print(f'FBR xong. Moi: {n_new} (fallback anh goc: {n_fail}) | Dung lai: {n_skip}')

In [ ]:
# ── Kiem tra bang mat: anh FBR co giong anh field khong? ──
_rng = random.Random(0)
n = 6
idxs = _rng.sample(range(len(fbr_samples)), n)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 6.5))
for j, idx in enumerate(idxs):
    axes[0, j].imshow(Image.open(pvd_samples[idx][0]).convert('RGB'))
    axes[0, j].set_title(f'PV {CLASS_NAMES[pvd_samples[idx][1]]}', fontsize=9); axes[0, j].axis('off')
    axes[1, j].imshow(Image.open(fbr_samples[idx][0]).convert('RGB'))
    axes[1, j].set_title('FBR', fontsize=9); axes[1, j].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for j, (p, l) in enumerate(_rng.sample(target_samples, 4)):
    axes[j].imshow(Image.open(p).convert('RGB'))
    axes[j].set_title(f'target ({CLASS_NAMES[l]})', fontsize=9); axes[j].axis('off')
plt.tight_layout(); plt.show()

### 4.1 — Loaders

- `SOURCE['fbr']` / `SOURCE['raw']`: cùng chỉ số train/val, chỉ khác ảnh (FBR hay gốc).
- `target_loader_3v`: `TargetThreeViewDataset` của notebook táo — trả `(plain, view1, view2)`;
  `plain` cho CDAN+E, hai view sinh trong `fork_rng` cho Mean Teacher. **Mọi** nhánh đều
  đọc loader này (kể cả nhánh không dùng target) để tiêu thụ RNG giống hệt nhau.

In [ ]:
class TargetThreeViewDataset(Dataset):
    # Target khong nhan -> ((plain, view1, view2), -1).
    # plain: VAL_TRANSFORM, tat dinh. view1/view2: TRAIN_TRANSFORM sinh trong fork_rng,
    # seed theo (base_seed, epoch, idx) -> khong dung RNG toan cuc, tai lap duoc sau resume.
    def __init__(self, samples, plain_transform, aug_transform, base_seed):
        self.samples         = samples
        self.plain_transform = plain_transform
        self.aug_transform   = aug_transform
        self.base_seed       = base_seed
        self.epoch           = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, _ = self.samples[idx]
        img   = Image.open(path).convert('RGB')
        plain = self.plain_transform(img)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed((self.base_seed * 1_000_003 + self.epoch * 10_007 + idx) % (2**31 - 1))
            v1 = self.aug_transform(img)
            v2 = self.aug_transform(img)
        return (plain, v1, v2), -1


def _source_loaders(samples):
    ds_tr = PlantDiseaseDataset(samples, CLASS_NAMES, TRAIN_TRANSFORM)
    ds_vl = PlantDiseaseDataset(samples, CLASS_NAMES, VAL_TRANSFORM)
    return (make_loader(Subset(ds_tr, src_tr_idx), BATCH_SIZE, shuffle=True, drop_last=True),
            make_loader(Subset(ds_vl, src_vl_idx), BATCH_SIZE, shuffle=False))

SOURCE = {'fbr': _source_loaders(fbr_samples), 'raw': _source_loaders(pvd_samples)}

dataset_tgt_3v   = TargetThreeViewDataset(target_samples, VAL_TRANSFORM, TRAIN_TRANSFORM, DATA_SEED)
target_loader_3v = make_loader(dataset_tgt_3v, BATCH_SIZE, shuffle=True, drop_last=True)
test_loader      = make_loader(PlantDiseaseDataset(test_samples, CLASS_NAMES, VAL_TRANSFORM),
                               BATCH_SIZE, shuffle=False)

# Chi dung khi co anh trung PV: field-only la tap con thuc su cua test.
field_test_loader = (make_loader(PlantDiseaseDataset(field_test_samples, CLASS_NAMES, VAL_TRANSFORM),
                                 BATCH_SIZE, shuffle=False)
                     if 0 < len(field_test_samples) < len(test_samples) else None)

assert len(target_loader_3v) >= 1, (
    f'Target {len(target_samples)} anh < BATCH_SIZE {BATCH_SIZE} -> giam BATCH_SIZE')

# ── Kiem tra tai cho (giong notebook tao) ───────────────────────────────────
(_p, _v1, _v2), _l = next(iter(target_loader_3v))
assert _p.shape == _v1.shape == _v2.shape == (BATCH_SIZE, 3, 224, 224)
assert not torch.allclose(_v1, _v2), 'view1 == view2 -> L_cons luon = 0'
assert not torch.allclose(_p, _v1), 'plain dang bi augment'
assert (_l == -1).all(), 'target loader tra ve nhan that'

torch.manual_seed(123); _before = torch.rand(3)
torch.manual_seed(123); dataset_tgt_3v.set_epoch(7); _ = dataset_tgt_3v[0]; _after = torch.rand(3)
assert torch.equal(_before, _after), 'fork_rng ro ri sang RNG toan cuc'
del _p, _v1, _v2, _l, _before, _after

print(f'source train: {len(SOURCE["fbr"][0])} batch/epoch | '
      f'target: {len(target_loader_3v)} batch/luot (lap lai khi het) | '
      f'test: {len(test_samples)} anh'
      + (f' | field-only: {len(field_test_samples)} anh' if field_test_loader is not None else ''))

## 5 — Models, CDAN+E & Mean Teacher helpers (giống notebook táo)

In [ ]:
def get_lambda(step, max_steps, gamma=10.0):
    p = step / max_steps
    return 2.0 / (1.0 + np.exp(-gamma * p)) - 1.0


def domain_lbl(n, is_src, device):
    return torch.full((n,), 0 if is_src else 1, dtype=torch.long, device=device)


class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None


class GradRevLayer(nn.Module):
    def __init__(self, lam=1.0):
        super().__init__()
        self.lam = lam
    def forward(self, x):
        return GRL.apply(x, self.lam)
    def set_lam(self, lam):
        self.lam = lam


class FeatureExtractor(nn.Module):
    # ResNet ImageNet, bo FC. BACKBONE (§2): resnet18 -> 512-d, resnet50 -> 2048-d (weights nhu notebook B).
    def __init__(self, pretrained=True, backbone=BACKBONE):
        super().__init__()
        if backbone == 'resnet18':
            net = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            self.out_dim = 512
        elif backbone == 'resnet50':
            net = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            self.out_dim = 2048
        else:
            raise ValueError(f'unknown backbone: {backbone}')
        self.net = nn.Sequential(*list(net.children())[:-1])
    def forward(self, x):
        return self.net(x).flatten(1)


class Classifier(nn.Module):
    def __init__(self, in_dim=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.fc(x)


class CDANDisc(nn.Module):
    # Input = multilinear map, dim = 512 * num_classes. Hai lop an 512, 256.
    def __init__(self, in_dim=512, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim * num_classes, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 2),
        )
    def forward(self, x):
        return self.net(x)


def multilinear_map(feat, pred):
    # KHONG detach feat: gradient doi khang phai qua GRL toi phi.
    return torch.bmm(feat.unsqueeze(2), pred.unsqueeze(1)).view(feat.size(0), -1)


def cdan_entropy_weight(probs):
    # w = 1 + exp(-H(p)); detach -> chi la he so.
    ent = -(probs * torch.log(probs + 1e-5)).sum(dim=1)
    return (1.0 + torch.exp(-ent)).detach()


# ── Mean Teacher ─────────────────────────────────────────────────────────────
EMA_DECAY     = 0.99
CONS_MAX      = float(NUM_CLASSES ** 2)   # = C^2 (notebook tao: 9 voi C=3)
CONS_START    = 100
CONS_RAMP_LEN = 50


def sigmoid_rampup(x, length):
    if length == 0:
        return 1.0
    x = float(np.clip(x, 0.0, length)) / length
    return float(np.exp(-5.0 * (1.0 - x) ** 2))


def consistency_weight(epoch, cons_max=CONS_MAX, start=CONS_START, ramp_len=CONS_RAMP_LEN):
    return 0.0 if epoch < start else cons_max * sigmoid_rampup(epoch - start, ramp_len)


def softmax_mse_loss(input_logits, target_logits):
    # Tong tren batch / num_classes; cho goi chia them batch size.
    return F.mse_loss(F.softmax(input_logits, 1), F.softmax(target_logits, 1),
                      reduction='sum') / input_logits.size(1)


def create_teacher(student_phi, student_G, device):
    phi_t = FeatureExtractor(pretrained=False).to(device)
    G_t   = Classifier(phi_t.out_dim, NUM_CLASSES).to(device)
    phi_t.load_state_dict(student_phi.state_dict())
    G_t.load_state_dict(student_G.state_dict())
    for p in list(phi_t.parameters()) + list(G_t.parameters()):
        p.requires_grad_(False)
    return phi_t, G_t


def update_ema(student_mods, teacher_mods, global_step, ema_decay=EMA_DECAY):
    # Chi .parameters(); BN buffer cua teacher cap nhat qua forward o .train().
    alpha = min(1 - 1 / (global_step + 1), ema_decay)
    with torch.no_grad():
        for sm, tm in zip(student_mods, teacher_mods):
            for ema_p, p in zip(tm.parameters(), sm.parameters()):
                ema_p.data.mul_(alpha).add_(p.data, alpha=1 - alpha)


# ── Danh gia: tong, theo lop, macro, confusion ──────────────────────────────
def evaluate(phi, G, loader, device):
    phi.eval(); G.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)   # [true, pred]
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = G(phi(imgs.to(device))).argmax(1).cpu().numpy()
            for t, p in zip(lbls.numpy(), preds):
                cm[t, p] += 1
    per_class = {c: (100.0 * cm[i, i] / cm[i].sum() if cm[i].sum() else float('nan'))
                 for i, c in enumerate(CLASS_NAMES)}
    return {
        'acc': 100.0 * np.trace(cm) / cm.sum(),
        'macro_acc': float(np.nanmean(list(per_class.values()))),
        'per_class': per_class,
        'confusion': cm.tolist(),
    }

print(f'Models & helpers defined. NUM_CLASSES={NUM_CLASSES}, CONS_MAX={CONS_MAX}')

## 6 — Một vòng lặp cho cả 5 nhánh

`train_arm(arm, seed)` là vòng lặp `train_cdane` của notebook táo, tổng quát hoá bằng
hai cờ `use_cdan` / `use_mt`. Giữ nguyên bốn điểm ghép cặp RNG:

1. Thứ tự khởi tạo `FeatureExtractor → Classifier → CDANDisc` ở **mọi** nhánh
   (nhánh không dùng D vẫn khởi tạo nó để tiêu thụ RNG giống nhau).
2. Teacher dựng trong `fork_rng` ở mọi nhánh.
3. `g.manual_seed(seed*1000 + epoch)` mỗi epoch.
4. View của target sinh trong `fork_rng`, seed theo `(DATA_SEED, epoch, lượt)`.

**Thay đổi duy nhất về cấu trúc:** một epoch = một lượt qua **source**. Target lớn hơn source nên mỗi epoch
chỉ đi qua một phần target (thứ tự xáo theo `g`); nếu target hết trước, nó được lặp lại và mỗi lượt lặp gọi
`set_epoch(epoch*100 + lượt)` để hai view không lặp lại y hệt trong cùng epoch.

Chọn model: `val_loss` trên source val, từ epoch `MIN_SEL` — đo bằng **student** cho
nhánh không có MT, **teacher (EMA)** cho nhánh có MT (giống notebook táo).

`probe_epochs=N` chạy **đúng vòng lặp này** nhưng bỏ forward/backward, chỉ ghi chữ ký
batch source — đó là cách cell `rng_probe` kiểm chứng ghép cặp mà không cần code riêng.

In [ ]:
ARMS = {
    #  arm        : (source, use_cdan, use_mt, ten hien thi)
    'src_raw'  : ('raw', False, False, 'Source-only (PV goc, no FBR)'),
    'src_fbr'  : ('fbr', False, False, 'Source-only w/ FBR'),
    'cdane'    : ('fbr', True,  False, 'CDAN+E w/ FBR'),
    'mt_only'  : ('fbr', False, True,  'MT only w/ FBR'),
    'cdane_mt' : ('fbr', True,  True,  'CDAN+E + MT w/ FBR'),
}
PAIRED_ARMS = ['src_fbr', 'cdane', 'mt_only', 'cdane_mt']   # cung source FBR -> phai cung dong RNG


def _target_batches(epoch):
    # Lap target loader cho du so batch cua source. Moi luot moi -> view moi.
    n_pass = 0
    while True:
        dataset_tgt_3v.set_epoch(epoch * 100 + n_pass)
        for batch in target_loader_3v:
            yield batch
        n_pass += 1


def save_ckpt(path, epoch, step, best_loss, best_state, phi, G, D, opt, sch, phi_t, G_t):
    torch.save({'epoch': epoch, 'step': step, 'best_loss': best_loss, 'best_state': best_state,
                'phi': phi.state_dict(), 'G': G.state_dict(), 'D': D.state_dict(),
                'optimizer': opt.state_dict(), 'scheduler': sch.state_dict(),
                # state_dict cua teacher gom ca BN buffer -> bat buoc luu de resume dung.
                'phi_t': phi_t.state_dict(), 'G_t': G_t.state_dict()}, path)


def load_ckpt(path, phi, G, D, opt, sch, phi_t, G_t):
    if not os.path.exists(path):
        return 1, 0, float('inf'), None
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    phi.load_state_dict(ck['phi']); G.load_state_dict(ck['G']); D.load_state_dict(ck['D'])
    opt.load_state_dict(ck['optimizer']); sch.load_state_dict(ck['scheduler'])
    phi_t.load_state_dict(ck['phi_t']); G_t.load_state_dict(ck['G_t'])
    return ck['epoch'] + 1, ck['step'], ck['best_loss'], ck['best_state']


def train_arm(arm, seed, max_epochs=MAX_EPOCHS, min_sel=MIN_SEL, ckpt_dir=CKPT_DIR,
              log_every=LOG_EVERY, probe_epochs=None, verbose=True):
    src_key, use_cdan, use_mt, _ = ARMS[arm]
    train_loader_src, val_loader_src = SOURCE[src_key]
    set_seed(seed)

    # (1) Thu tu khoi tao giong nhau o moi nhanh.
    phi = FeatureExtractor(pretrained=True).to(DEVICE)
    G   = Classifier(phi.out_dim, NUM_CLASSES).to(DEVICE)
    D   = CDANDisc(phi.out_dim, NUM_CLASSES).to(DEVICE)
    grl = GradRevLayer()
    # (2) Teacher trong fork_rng.
    with torch.random.fork_rng(devices=[]):
        phi_t, G_t = create_teacher(phi, G, DEVICE)

    params = list(phi.parameters()) + list(G.parameters()) + (list(D.parameters()) if use_cdan else [])
    opt = optim.AdamW(params, lr=LR, weight_decay=WD)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    ce_mean, ce_persam = nn.CrossEntropyLoss(), nn.CrossEntropyLoss(reduction='none')

    n_epochs = probe_epochs or max_epochs
    ckpt_path = best_path = None
    start_epoch, step, best_loss, best_state = 1, 0, float('inf'), None
    if ckpt_dir is not None and probe_epochs is None:
        ckpt_path = os.path.join(ckpt_dir, f'{arm}_s{seed}_ckpt.pth')
        best_path = os.path.join(ckpt_dir, f'{arm}_s{seed}_best.pth')
        start_epoch, step, best_loss, best_state = load_ckpt(
            ckpt_path, phi, G, D, opt, sch, phi_t, G_t)
        if start_epoch > 1 and verbose:
            print(f'  [resume] {arm} s{seed}: tiep tuc tu epoch {start_epoch}')

    steps_per_epoch = len(train_loader_src)
    total_steps     = max_epochs * steps_per_epoch
    sel_phi, sel_G  = (phi_t, G_t) if use_mt else (phi, G)
    collapse_share  = 100 * (0.5 + 0.5 / NUM_CLASSES)
    sigs, lam       = [], 0.0

    for epoch in range(start_epoch, n_epochs + 1):
        g.manual_seed(seed * 1000 + epoch)                    # (3)
        tgt_iter = _target_batches(epoch)                     # (4) ben trong
        phi.train(); G.train(); D.train()
        phi_t.train(); G_t.train()                            # BN buffer cua teacher
        w_cons = consistency_weight(epoch) if use_mt else 0.0
        ep = {'cls': 0.0, 'cdan': 0.0, 'cons': 0.0}
        tgt_hist = torch.zeros(NUM_CLASSES, dtype=torch.long)

        for si, sl in train_loader_src:
            (t_plain, t_v1, t_v2), _ = next(tgt_iter)        # moi nhanh deu doc target
            if probe_epochs is not None:
                sigs.append((round(float(si.sum()), 4), int(sl.sum())))
                continue

            si, sl = si.to(DEVICE), sl.to(DEVICE)
            lam = get_lambda(step, total_steps); grl.set_lam(lam)
            step += 1
            opt.zero_grad()

            sf = phi(si)
            logits_s = G(sf)
            L_cls = ce_mean(logits_s, sl)
            total = L_cls

            if use_cdan:
                t_plain = t_plain.to(DEVICE)
                tf = phi(t_plain)
                logits_t = G(tf)
                sp, tp = torch.softmax(logits_s, 1), torch.softmax(logits_t, 1)
                src_mm = grl(multilinear_map(sf, sp.detach()))
                tgt_mm = grl(multilinear_map(tf, tp.detach()))
                sw = cdan_entropy_weight(sp); sw = sw / sw.sum()
                tw = cdan_entropy_weight(tp); tw = tw / tw.sum()
                src_d = (sw * ce_persam(D(src_mm), domain_lbl(si.size(0), True, DEVICE))).sum()
                tgt_d = (tw * ce_persam(D(tgt_mm), domain_lbl(t_plain.size(0), False, DEVICE))).sum()
                L_cdan = (src_d + tgt_d) / 2.0
                total = total + L_cdan
                ep['cdan'] += L_cdan.item()
                probe = logits_t.detach()

            if use_mt:
                t_v1, t_v2 = t_v1.to(DEVICE), t_v2.to(DEVICE)
                student_logits = G(phi(t_v1))
                with torch.no_grad():
                    teacher_logits = G_t(phi_t(t_v2))
                L_cons = softmax_mse_loss(student_logits, teacher_logits) / t_v1.size(0)
                total = total + w_cons * L_cons
                ep['cons'] += L_cons.item()
                probe = teacher_logits

            if use_cdan or use_mt:
                tgt_hist += torch.bincount(probe.argmax(1).cpu(), minlength=NUM_CLASSES)

            total.backward()
            opt.step()
            if use_mt:
                update_ema([phi, G], [phi_t, G_t], step)
            ep['cls'] += L_cls.item()

        if probe_epochs is not None:
            continue
        sch.step()

        # Validation tren source val (khong dung nhan target).
        sel_phi.eval(); sel_G.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for vi, vlab in val_loader_src:
                vi, vlab = vi.to(DEVICE), vlab.to(DEVICE)
                logits = sel_G(sel_phi(vi))
                vl += ce_mean(logits, vlab).item()
                vc += (logits.argmax(1) == vlab).sum().item()
                vt += vlab.size(0)
        val_loss, val_acc = vl / len(val_loader_src), vc / vt

        if epoch >= min_sel and val_loss < best_loss:
            best_loss  = val_loss
            best_state = {'phi': copy.deepcopy(sel_phi.state_dict()),
                          'G':   copy.deepcopy(sel_G.state_dict())}
            if best_path:
                torch.save(best_state, best_path)

        if verbose and (epoch % log_every == 0 or epoch == max_epochs):
            msg = (f'  [{arm} s{seed}] ep {epoch:03d}/{max_epochs} val_acc={val_acc*100:5.1f}% '
                   f'lam={lam:.3f} w={w_cons:.2f} | cls={ep["cls"]/steps_per_epoch:.3f} '
                   f'cdan={ep["cdan"]/steps_per_epoch:.3f} cons={ep["cons"]/steps_per_epoch:.4f}')
            if tgt_hist.sum() > 0:
                share = tgt_hist.float() / tgt_hist.sum() * 100
                msg += ' | tgt%=[' + ','.join(f'{s:.0f}' for s in share) + ']'
                if share.max().item() > collapse_share:
                    msg += f'\n     CANH BAO: target du doan don ve 1 lop ({share.max():.0f}%)'
            print(msg)

        if ckpt_path and (epoch % log_every == 0 or epoch == max_epochs):
            save_ckpt(ckpt_path, epoch, step, best_loss, best_state, phi, G, D, opt, sch, phi_t, G_t)

    if probe_epochs is not None:
        return sigs

    if best_state is None:
        print(f'  CANH BAO [{arm}]: chua toi cua so chon model, dung weight cuoi.')
        best_state = {'phi': copy.deepcopy(sel_phi.state_dict()), 'G': copy.deepcopy(sel_G.state_dict())}

    eval_phi = FeatureExtractor(pretrained=False).to(DEVICE)
    eval_G   = Classifier(eval_phi.out_dim, NUM_CLASSES).to(DEVICE)
    eval_phi.load_state_dict(best_state['phi']); eval_G.load_state_dict(best_state['G'])

    res = evaluate(eval_phi, eval_G, test_loader, DEVICE)
    res.update({'arm': arm, 'seed': seed, 'data_seed': DATA_SEED, 'task': TASK,
                'protocol': PROTOCOL, 'backbone': BACKBONE, 'n_test': len(test_samples),
                'best_val_loss': best_loss, 'epochs_run': max_epochs,
                'acc_student_final': evaluate(phi, G, test_loader, DEVICE)['acc']})
    if field_test_loader is not None:
        # Test bo anh trung PV (§3.3): so phu, khong co trong bai bao.
        res['field_only'] = evaluate(eval_phi, eval_G, field_test_loader, DEVICE)
        res['n_field_only'] = len(field_test_samples)
    if use_mt and res['acc_student_final'] > res['acc'] + 3.0:
        print(f'  CANH BAO: student ({res["acc_student_final"]:.2f}%) hon han teacher '
              f'({res["acc"]:.2f}%) - nghi ngo BN buffer chua sync')
    return res

print('train_arm() defined. Arms:', list(ARMS))

## 7 — Kiểm chứng ghép cặp RNG (chạy trước, vài phút, không huấn luyện)

Bốn nhánh dùng source FBR phải thấy **cùng** chuỗi ảnh source (cùng thứ tự, cùng
augmentation). `[FAIL]` nghĩa là chênh lệch accuracy giữa các nhánh không còn quy được
về loss — đừng chạy 300 epoch.

In [ ]:
_ref = train_arm(PAIRED_ARMS[0], SEEDS[0], probe_epochs=2, verbose=False)
print(f'So batch source doi chieu: {len(_ref)}')
for _arm in PAIRED_ARMS[1:]:
    _sig = train_arm(_arm, SEEDS[0], probe_epochs=2, verbose=False)
    if _sig == _ref:
        print(f'  [OK]   {PAIRED_ARMS[0]} == {_arm}')
    else:
        _i = next(i for i, (a, b) in enumerate(zip(_ref, _sig)) if a != b)
        print(f'  [FAIL] {PAIRED_ARMS[0]} vs {_arm}: lech tu batch {_i}: {_ref[_i]} vs {_sig[_i]}')
    assert _sig == _ref, f'{_arm} KHONG ghep cap voi {PAIRED_ARMS[0]} - dung chay tiep.'
del _ref, _sig

## 8 — Chạy tất cả nhánh × seed

Mỗi `(nhánh, seed)` ghi một file JSON trong `RESULTS_DIR`; đã có thì bỏ qua (xoá file
để chạy lại). Colab ngắt giữa chừng thì chạy lại cell này — checkpoint mỗi
`LOG_EVERY` epoch, tự resume.

Muốn chạy bớt nhánh, sửa `RUN_ARMS`. Thời gian tham khảo trên T4: nhánh `src_*` nhanh
nhất, `cdane_mt` chậm nhất (4 forward pass / iteration).

In [ ]:
# src_raw = dong doi chieu voi 'Baseline (w/o DA)' cua MSUN (khong thich nghi, khong FBR).
RUN_ARMS = ['cdane_mt']   # du ablation: list(ARMS)


def result_path(arm, seed):
    return os.path.join(RESULTS_DIR, f'{arm}_s{seed}.json')


for seed in SEEDS:
    for arm in RUN_ARMS:
        path = result_path(arm, seed)
        if os.path.exists(path):
            print(f'[skip] {os.path.basename(path)} da co')
            continue
        print('=' * 64)
        print(f'ARM: {ARMS[arm][3]}  (seed={seed}, task={TASK})')
        print('=' * 64)
        res = train_arm(arm, seed)
        with open(path, 'w') as f:
            json.dump(res, f, indent=2)
        pc = ', '.join(f'{c}={v:.2f}%' for c, v in res['per_class'].items())
        print(f'-> test target: acc={res["acc"]:.2f}%  macro={res["macro_acc"]:.2f}%  ({pc})')
    if 'field_only' in res:
        print(f'   field-only ({res["n_field_only"]} anh): acc={res["field_only"]["acc"]:.2f}%  '
              f'macro={res["field_only"]["macro_acc"]:.2f}%')

## 9 — So sánh

Bảng 1: các nhánh của ta (mean ± std qua seed); cột `field_acc` chỉ xuất hiện khi target có ảnh trùng PV. Bảng 2: chênh lệch
**ghép cặp** giữa các nhánh. Bảng 3: đặt cạnh bảng kết quả của bài báo, theo từng lớp.

In [ ]:
# === CELL-TAG: compare ===
_rows = {}
for arm in ARMS:
    rs = [json.load(open(result_path(arm, s))) for s in SEEDS if os.path.exists(result_path(arm, s))]
    if rs:
        _rows[arm] = rs

if not _rows:
    print('Chua co ket qua nao. Chay cell muc 8 truoc.')
else:
    def _ms(vals):
        return f'{np.mean(vals):6.2f}' + (f' ± {np.std(vals, ddof=1):5.2f}' if len(vals) > 1 else '        ')

    _has_field = any('field_only' in r for rs in _rows.values() for r in rs)
    cols = ['acc', 'macro_acc'] + (['field_acc'] if _has_field else []) + CLASS_NAMES
    w = 18
    print(f'BANG 1 - cac nhanh ({PROTOCOL}, {BACKBONE}, test = {len(test_samples)} anh target)')
    print('=' * (36 + w * len(cols)))
    print(f'{"Method":<30}{"n":>4}  ' + ''.join(f'{c:>{w}}' for c in cols))
    print('-' * (36 + w * len(cols)))
    for arm, rs in _rows.items():
        vals = [_ms([r['acc'] for r in rs]), _ms([r['macro_acc'] for r in rs])]
        if _has_field:
            vals.append(_ms([r['field_only']['acc'] for r in rs if 'field_only' in r]))
        vals += [_ms([r['per_class'][c] for r in rs]) for c in CLASS_NAMES]
        print(f'{ARMS[arm][3]:<30}{len(rs):>4}  ' + ''.join(f'{v:>{w}}' for v in vals))
    print('=' * (36 + w * len(cols)))

    # ── Chenh lech ghep cap (cung seed) ─────────────────────────────────────
    PAIRS = [('src_fbr', 'src_raw', 'FBR'),
             ('cdane', 'src_fbr', 'CDAN+E (tren FBR)'),
             ('mt_only', 'src_fbr', 'MT (tren FBR)'),
             ('cdane_mt', 'cdane', 'them MT vao CDAN+E'),
             ('cdane_mt', 'mt_only', 'them CDAN+E vao MT')]
    print('\nBANG 2 - CHENH LECH GHEP CAP (cung seed), don vi: diem %')
    print('-' * 72)
    for a, b, label in PAIRS:
        if a in _rows and b in _rows:
            ra = {r['seed']: r for r in _rows[a]}
            rb = {r['seed']: r for r in _rows[b]}
            common = sorted(set(ra) & set(rb))
            if not common:
                continue
            d_acc   = [ra[s]['acc'] - rb[s]['acc'] for s in common]
            d_macro = [ra[s]['macro_acc'] - rb[s]['macro_acc'] for s in common]
            print(f'  {label:<22} {a:>9} - {b:<8}: acc {_ms(d_acc)} | macro {_ms(d_macro)}  (n={len(common)})')

    # ── Dat canh bang cua bai bao ────────────────────────────────────────────
    cw = max(11, max(map(len, CLASS_NAMES)) + 2)
    print(f'\nBANG 3 - DAT CANH {REF_TABLE_NAME} (accuracy % theo lop; cot cuoi = so tong bai bao bao cao)')
    print('-' * (44 + cw * (NUM_CLASSES + 1)))
    print(f'  {"Method":<42}' + ''.join(f'{c:>{cw}}' for c in CLASS_NAMES) + f'{"tong":>{cw}}')
    print('-' * (44 + cw * (NUM_CLASSES + 1)))
    for m, (per_class, total) in REF_RESULTS.items():
        print(f'  {m:<42}' + ''.join(f'{v:>{cw}.2f}' for v in per_class) + f'{total:>{cw}.2f}')
    print('-' * (44 + cw * (NUM_CLASSES + 1)))
    for arm, rs in _rows.items():
        pc = [np.mean([r['per_class'][c] for r in rs]) for c in CLASS_NAMES]
        print(f'  {ARMS[arm][3] + " [ours]":<42}' + ''.join(f'{v:>{cw}.2f}' for v in pc)
              + f'{np.mean([r["acc"] for r in rs]):>{cw}.2f}')
    print('-' * (44 + cw * (NUM_CLASSES + 1)))
    print(f'  Target: bai bao {sum(REF_TARGET_COUNTS.values())} anh '
          f'({", ".join(f"{c}={n}" for c, n in REF_TARGET_COUNTS.items())}); '
          f'ours {len(test_samples)} anh ({_count(test_samples)}).')
    print('  Cot "tong" cua bai bao KHONG phai trung binh cac cot lop (khong tinh lai duoc tu bang);')
    print('  so sanh bang cot lop va accuracy tong cua ta. Dong doi chieu cong bang nhat: src_raw vs baseline.')

    # ── Confusion matrix cua seed dau tien ──────────────────────────────────
    print(f'\nCONFUSION MATRIX (seed {SEEDS[0]}, hang = nhan that, cot = du doan; thu tu {CLASS_NAMES})')
    for arm, rs in _rows.items():
        r0 = next((r for r in rs if r['seed'] == SEEDS[0]), rs[0])
        print(f'  {ARMS[arm][3]:<26} {r0["confusion"]}')

### Cách phát biểu kết quả trong luận văn

1. **Nêu số ảnh trùng PV in ở §3.3 trước mọi kết quả.** Nếu phần lớn CLD là ảnh PlantVillage thì PVD → CLD **không phải** bài
   toán lab → field, và 96.78% của MSUN cũng mang lỗi đó. Phát biểu: *"Trên toàn bộ CLD (giao thức MSUN), pipeline đạt X% so với
   96.78% của MSUN; trên N ảnh CLD không trùng PlantVillage, pipeline đạt Y%."*
2. Con số field-only **không có đối chứng** trong bài báo; dùng nó cho phân tích, không dùng để xếp hạng với MSUN.
3. **Source khác.** MSUN dùng PV ngô 411 / 788 / 930 / 954; ta dùng 300 ảnh mỗi lớp, cân bằng.
4. **Cột Average của MSUN không phải trung bình lớp** (lớp của MSUN trung bình 89.67 nhưng bài ghi 96.78). So từng lớp.
5. Kết luận về từng thành phần dùng bảng ghép cặp (Bảng 2). Nên chạy `SEEDS = [42, 43, 44]`.